# 🔥 Project 19 — Detect Burned Areas

## 🎯 Goal

Compare satellite images taken **before and after a wildfire** to identify burned areas.

We'll use the **Normalized Burn Ratio (NBR)** and compare it between two dates.

---

## 🌍 Why is this useful in Remote Sensing?

Burned area mapping is used for:

- 🔥 Wildfire damage assessment
- 🌲 Forest monitoring
- 🌱 Vegetation recovery studies
- 🌍 Climate change research
- 🚒 Disaster management

Government agencies often create burned area maps within days after a wildfire to estimate the affected area.

---

# ⭐ New Skill: Burn Severity Analysis

Previously (Project 10):

```text
Before Rain
      ↓
NDWI
      ↓

After Rain
      ↓
NDWI
      ↓

Subtract
      ↓
Water Change
```

Today:

```text
Before Fire
      ↓
NBR
      ↓

After Fire
      ↓
NBR
      ↓

Subtract
      ↓
Burned Area Map
```

Notice the workflow is almost identical.

Only the **index** changes.

---

# 🔗 Connect to Previous Lessons

You already know:

| Project | Index | Purpose |
|---------|-------|---------|
| 6 | NDVI | Vegetation |
| 9 | NDWI | Water |
| 10 | NDWI Difference | Water Change |
| 19 | NBR | Burned Areas |

You're building on the same ideas, not learning an entirely new workflow.

---

# 📖 What is NBR?

The **Normalized Burn Ratio (NBR)** is an index specifically designed to detect burned vegetation.

It uses:

- **B8 (NIR)** → Healthy vegetation reflects strongly.
- **B12 (SWIR2)** → Burned and dry surfaces reflect strongly.

### Formula

```text
        NIR - SWIR
NBR = -------------
        NIR + SWIR
```

For Sentinel-2:

```text
NIR  = B8
SWIR = B12
```

Healthy vegetation usually has a **high NBR**.

After a fire, **NBR decreases** because vegetation is destroyed.

---

# Step 1 — Define the Study Area

We'll use a study area that experienced a wildfire (or any area for practice).

```javascript
var studyArea = ee.Geometry.Polygon([
  [
    [13.20,52.60],
    [13.65,52.60],
    [13.65,52.35],
    [13.20,52.35],
    [13.20,52.60]
  ]
]);
```

---

# Step 2 — Load the Sentinel-2 Collection

We'll create a collection.

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);
```

Just like Project 10, we'll select two images from different dates.

---

# Step 3 — Create an NBR Function ⭐

```javascript
function calculateNBR(image){

  return image
    .normalizedDifference(['B8','B12'])
    .rename('NBR');

}
```

This should look familiar.

Previously:

```text
calculateNDVI()
calculateNDWI()
```

Now:

```text
calculateNBR()
```

Only the bands have changed.

---

# Step 4 — Select the Before-Fire Image

```javascript
var beforeFire = collection
  .filterBounds(studyArea)
  .filterDate('2024-06-01','2024-06-30')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(studyArea);
```

---

# Step 5 — Select the After-Fire Image

```javascript
var afterFire = collection
  .filterBounds(studyArea)
  .filterDate('2024-08-01','2024-08-31')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(studyArea);
```

> **Note:** For a real wildfire study, choose dates immediately before and after a known fire event. Here we're simply using two different periods to practice the workflow.

---

# Step 6 — Calculate NBR

```javascript
var beforeNBR = calculateNBR(beforeFire);

var afterNBR = calculateNBR(afterFire);
```

Exactly like Project 10.

---

# Step 7 — Calculate Burn Change ⭐

```javascript
var burnChange = beforeNBR.subtract(afterNBR);
```

Notice the order.

We use:

```text
Before - After
```

Why?

Suppose one pixel has:

| Date | NBR |
|------|----:|
| Before | 0.70 |
| After | 0.20 |

Earth Engine calculates:

```text
0.70 - 0.20 = 0.50
```

A **large positive value** indicates that vegetation decreased after the fire, suggesting a burned area.

---

# Step 8 — Visualization

```javascript
var burnVis = {
  min:0,
  max:0.6,
  palette:[
    'white',
    'yellow',
    'orange',
    'red'
  ]
};
```

### Interpretation

- ⚪ White → Little or no change
- 🟡 Yellow → Slight burn
- 🟠 Orange → Moderate burn
- 🔴 Red → Severe burn

---

# Step 9 — Display

```javascript
Map.centerObject(studyArea,10);

Map.addLayer(
  burnChange,
  burnVis,
  'Burned Areas'
);
```

Areas with stronger red colors indicate larger decreases in NBR, which usually correspond to more severely burned vegetation.

---

# 💻 Complete Code

```javascript
// =====================================
// 1. Study Area
// =====================================
var studyArea = ee.Geometry.Polygon([
  [
    [13.20,52.60],
    [13.65,52.60],
    [13.65,52.35],
    [13.20,52.35],
    [13.20,52.60]
  ]
]);

// =====================================
// 2. Image Collection
// =====================================
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);

// =====================================
// 3. NBR Function
// =====================================
function calculateNBR(image){

  return image
    .normalizedDifference(['B8','B12'])
    .rename('NBR');

}

// =====================================
// 4. Before Fire
// =====================================
var beforeFire = collection
  .filterBounds(studyArea)
  .filterDate('2024-06-01','2024-06-30')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(studyArea);

// =====================================
// 5. After Fire
// =====================================
var afterFire = collection
  .filterBounds(studyArea)
  .filterDate('2024-08-01','2024-08-31')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(studyArea);

// =====================================
// 6. Calculate NBR
// =====================================
var beforeNBR = calculateNBR(beforeFire);
var afterNBR = calculateNBR(afterFire);

// =====================================
// 7. Burn Change
// =====================================
var burnChange = beforeNBR.subtract(afterNBR);

// =====================================
// 8. Visualization
// =====================================
var burnVis = {
  min:0,
  max:0.6,
  palette:[
    'white',
    'yellow',
    'orange',
    'red'
  ]
};

// =====================================
// 9. Display
// =====================================
Map.centerObject(studyArea,10);

Map.addLayer(
  burnChange,
  burnVis,
  'Burned Areas'
);
```

---

# 📌 Things to Remember

## 1. NBR

NBR measures burned vegetation.

```text
B8 (NIR)
      +
B12 (SWIR)
      ↓
NBR
```

It is specifically designed for wildfire analysis.

---

## 2. Burn Change

The workflow is:

```text
Before Fire
      ↓
NBR
      ↓

After Fire
      ↓
NBR
      ↓

Before - After
      ↓
Burn Change
```

This is the same change detection workflow you learned in Project 10.

---

## 3. Why `before.subtract(after)`?

Healthy vegetation usually has a higher NBR before a fire.

Subtracting:

```text
Before - After
```

produces **positive values** where vegetation was lost due to burning.

---

# ⚠️ Common Beginner Mistakes

### ❌ Mistake 1: Using NDVI instead of NBR

NDVI measures vegetation health, but **NBR** is specifically designed to detect burned areas because it uses the **SWIR band**, which is highly sensitive to fire damage.

---

### ❌ Mistake 2: Reversing the subtraction

```javascript
afterNBR.subtract(beforeNBR)
```

This flips the sign of the results and makes interpretation more confusing.

---

### ❌ Mistake 3: Choosing dates unrelated to a fire

For meaningful results, the images should be taken as close as possible to the wildfire event while avoiding excessive cloud cover.

---

# 📝 Your Practice (Project 19.1)

Create a burned area map for a wildfire (or any study area for practice).

### Requirements

1. Create a study area.
2. Load the Sentinel-2 collection.
3. Write a `calculateNBR()` function.
4. Select a before-fire image.
5. Select an after-fire image.
6. Calculate NBR for both images.
7. Create a burn change image using:

```javascript
beforeNBR.subtract(afterNBR)
```

8. Display the burned area map.

---

## Before writing the code, answer these two questions.

### Question 1

Why does NBR use:

```text
B8 (NIR)

and

B12 (SWIR)
```

instead of the NDVI bands (**B8 and B4**)?

---

### Question 2

Why do we calculate:

```javascript
beforeNBR.subtract(afterNBR)
```

instead of subtracting the original Sentinel-2 images directly?